# Fresh AI: train `fresh-unified-1` (run 3: fixed weight initialization)

First real training run of Fresh's **own** model on Fresh's authorized, screened corpus (oasst1, Dolly, Swahili + English Wikipedia, Fresh's own data). From-scratch weights, no outside model or pretrained vocabulary.

**Before you start:** Runtime menu → *Change runtime type* → **T4 GPU**. Then run the cells top to bottom. Everything is saved to your Google Drive, so if Colab disconnects, reconnect and re-run all cells: finished steps are skipped and training **resumes**.

**What to expect (honest):** ~50M parameters on ~35M tokens is a small model. It will learn language structure and some Fresh/Swahili/English knowledge, but it is *not* superintelligence and it will make mistakes. This is stage 1 of the Beyond-Superintelligence roadmap (truth, memory, tools, governance come from the platform around the model). The checkpoint is never marked production-ready by this notebook.

In [ ]:
# 1. Setup: GPU check, Google Drive, repo, packages
import subprocess, os
print(subprocess.run(["nvidia-smi","-L"],capture_output=True,text=True).stdout or "NO GPU: switch runtime to T4 GPU")
from google.colab import drive
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/fresh-ai/run2"
os.makedirs(ROOT, exist_ok=True)
if not os.path.exists("/content/fresh-web-lite"):
    subprocess.run(["git","clone","--depth","1","https://github.com/kanarafat54-boop/fresh-web-lite.git","/content/fresh-web-lite"],check=True)
%cd /content/fresh-web-lite
!pip -q install tokenizers numpy
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())

In [ ]:
# 2. Build the training mix (every row re-checked against the authorization registry)
!ls {ROOT}/mix/train.jsonl.gz 2>/dev/null || python3 training/build_training_mix.py --recipe training/mix_recipe_v1.json --out-dir {ROOT}/mix | head -20
!ls -lh {ROOT}/mix

In [ ]:
# 3. Train Fresh's tokenizer (byte-level BPE learned only from the mix). Takes a few minutes on CPU.
!ls {ROOT}/tokenizer-v2.json 2>/dev/null || python3 training/fresh_tokenizer_v2.py train --mix-train {ROOT}/mix/train.jsonl.gz --output {ROOT}/tokenizer-v2.json

In [ ]:
# 4. Turn text into token streams
!ls {ROOT}/train.bin 2>/dev/null || python3 training/fresh_tokenizer_v2.py encode --tokenizer {ROOT}/tokenizer-v2.json --input {ROOT}/mix/train.jsonl.gz --output {ROOT}/train.bin
!ls {ROOT}/validation.bin 2>/dev/null || python3 training/fresh_tokenizer_v2.py encode --tokenizer {ROOT}/tokenizer-v2.json --input {ROOT}/mix/validation.jsonl.gz --output {ROOT}/validation.bin

In [ ]:
# 5. SMOKE TEST (about 1 minute): tiny model, 20 steps. If this fails, send Claude the error before the long run.
!python3 training/train_fresh_unified_full.py --train-bin {ROOT}/train.bin --val-bin {ROOT}/validation.bin --tokenizer {ROOT}/tokenizer-v2.json \
  --spec training/model_spec_smoke.json --seq-len 256 --micro-batch 4 --accum 1 --max-steps 20 --warmup 5 --eval-every 10 --save-every 20 --eval-blocks 8 \
  --out-dir /content/smoke --run-id smoke 2>&1 | tail -25

In [ ]:
# 6. REAL TRAINING (about 4.5 hours on a free T4; resumes automatically if interrupted).
# If you hit 'out of memory', change --micro-batch 8 to 4 and --accum 8 to 16, then run again.
!python3 training/train_fresh_unified_full.py --train-bin {ROOT}/train.bin --val-bin {ROOT}/validation.bin --tokenizer {ROOT}/tokenizer-v2.json \
  --spec training/model_spec.json --seq-len 1024 --micro-batch 8 --accum 8 --epochs 3 \
  --mix-manifest {ROOT}/mix/manifest.json --out-dir {ROOT}/train-run3 --run-id fresh-unified-1-run3

In [ ]:
# 7. Results: send Claude this output (the manifest and samples are the evidence)
!cat {ROOT}/train-run3/fresh-unified-1-run3.manifest.json
!cat {ROOT}/train-run3/fresh-unified-1-run3.samples.txt
!ls -lh {ROOT}/train-run3